In [4]:
# Importing libraries
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split , GridSearchCV # GridSearchCV for HyperParameter tuning
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight


In [12]:
# Loading Data

df = pd.read_csv('/content/diabetes.csv')

In [13]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [14]:
# Basic Cleaning
# Removing unrealistic feautures
df = df.drop(columns = ["SkinThickness", "DiabetesPedigreeFunction"])

# Replace Impossible Zero
cols_with_zero_issue =   ["Glucose","BloodPressure","Insulin","BMI"]

for col in cols_with_zero_issue:
  df[col] = df[col].replace(0,np.nan)


In [15]:
df = df.drop(columns =["Pregnancies"])

In [16]:
# Target Split

X = df.drop("Outcome" , axis = 1)
y = df["Outcome"]

In [17]:
X

,Glucose,BloodPressure,Insulin,BMI,Age
0,148.0,72.0,NaN,33.6,50
1,85.0,66.0,NaN,26.6,31
2,183.0,64.0,NaN,23.3,32
3,89.0,66.0,94.0,28.1,21
4,137.0,40.0,168.0,43.1,33
...,...,...,...,...,...
763,101.0,76.0,180.0,32.9,63
764,122.0,70.0,NaN,36.8,27
765,121.0,72.0,112.0,26.2,30
766,126.0,60.0,NaN,30.1,47


In [19]:
X_train,X_test,y_train,y_test = train_test_split(
    X,y,
    test_size = 0.2,
    random_state=42,
    stratify=y
)

In [20]:
# Data PreProcessing
numeric_features = X.columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer" , SimpleImputer(strategy="median")),
    ("scaler" , StandardScaler())
])

preprocessor = ColumnTransformer([
    ("num",numeric_pipeline,numeric_features)
])

In [21]:
# Handling Class Imbalance
class_weights = compute_class_weight(
    class_weight = "balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weight_dict=dict(zip(np.unique(y_train),class_weights))

In [22]:
# Model Building

rf = RandomForestClassifier(
    class_weight=class_weight_dict,
    random_state=42
)

In [23]:
pipeline = Pipeline([
    ("preprocessor" , preprocessor),
    ("classifier" , rf)
])

In [26]:
# HyperPrameter Tuning
param_grid = {
    "classifier__n_estimators": [200, 300],
    "classifier__max_depth": [None, 5, 10],
    "classifier__min_samples_split": [2, 5]
}


In [27]:
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

In [28]:
grid.fit(X_train,y_train)

best_model = grid.best_estimator_

In [29]:
 # Evaluation
y_pred = best_model.predict(X_test)
y_proba= best_model.predict_proba(X_test)[:,1]

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("ROC-AUC Score:", roc_auc_score(y_test, y_proba))


Classification Report:

              precision    recall  f1-score   support

           0       0.86      0.73      0.79       100
           1       0.61      0.78      0.68        54

    accuracy                           0.75       154
   macro avg       0.73      0.75      0.74       154
weighted avg       0.77      0.75      0.75       154

ROC-AUC Score: 0.8201851851851851


In [31]:
# ==============================
# 11. Save Model
# ==============================
with open("/content/diabetes_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

print("Model saved successfully.")

Model saved successfully.
